# Analysis

In [ ]:
from tools.utils import replace_minus_ones_with_prev
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import seaborn as sns


In [ ]:
labeldist = []
for cl in ['opt-out','arbitration','class waiver','anti-scraping', 'modification']:

    df = pd.read_csv(f'annotations/ollama_annotations/{cl}_labels_judged.csv', index_col=0)
    labeldist.append(df['label_gpt-4o'].value_counts())

In [ ]:
pd.concat(labeldist, axis=1, keys=['opt-out','arbitration','class waiver','anti-scraping', 'modification']).fillna(0).sum(axis=0)

In [ ]:
processed_data_dir = Path('processed_data/tous')
file_name = processed_data_dir / 'metadata_annotated_distilbert.tsv' # _annotated_distilbert
metadata = pd.read_csv(file_name,sep='\t')
metadata['year_int'] = metadata.year.apply(lambda x: int(str(x)[:4]))
metadata.fillna('', inplace=True)

In [ ]:

threshold = 0.9
for clause_type in ['opt-out','arbitration','class waiver','anti-scraping', 'modification']:
    metadata[f'{clause_type}_recoded'] = 0
    metadata.loc[metadata[f'prob_1_{clause_type}'] >= threshold, f'{clause_type}_recoded'] = 1
    

In [ ]:
for clause_type in ['arbitration','class waiver','modification']: # 'opt-out','arbitration','class waiver','anti-scraping', 'modification'
    metadata.groupby('year_int')[f'{clause_type}_recoded'].sum().plot(legend=True, label=clause_type, figsize=(6,6))

In [ ]:
clause_type = 'modification' # 'opt-out' | 'arbitration' | 'class waiver' | 'anti-scraping' | 'modification'
metadata.groupby(f'{clause_type}_recoded').size()

In [ ]:
metadata[metadata.modification_recoded == 1].sentence

In [ ]:

data = metadata.groupby(['platform','year_int'])[f'{clause_type}_recoded'].sum().astype(bool).astype(int).unstack().fillna(-1)
#data = metadata.groupby(['platform','year_int'])[f'prob_1_{clause_type}'].max().unstack().fillna(-1)


In [ ]:
#data = data.loc[(data > -1).sum(axis=1).sort_values(ascending=False).index]

In [ ]:
sns.heatmap(data,cbar=False, cmap='YlGnBu', linewidths=.5, linecolor='gray', vmin=-1, vmax=1)

In [ ]:
sns.set(rc={'figure.figsize':(10.7,10.27)})
data_replaced = replace_minus_ones_with_prev(data)
data_replaced = data_replaced.loc[(data_replaced > -1).sum(axis=1).sort_values(ascending=False).index]
data_replaced.to_csv(f'results/{clause_type}_heatmap_data.csv')
sns.heatmap(data_replaced,cbar=False, cmap='YlGnBu', linewidths=.5, linecolor='gray', vmin=-1, vmax=1)

In [ ]:
# clause_data = metadata[metadata[f'{clause_type}_recoded'] == 1]
# index = clause_data.sort_values('year_int').index

In [ ]:
embeddings = np.loadtxt('processed_data/tous/embedding.tsv')

In [ ]:
# clause_embeddings = embeddings[index]
# clause_embeddings.shape[0], len(clause_data)

In [ ]:
# import scipy.spatial as sp
# from matplotlib import pyplot as plt
# sns.set()

# mult = 1 - sp.distance.cdist(clause_embeddings, clause_embeddings, 'cosine')
# sns.set(rc={'figure.figsize':(7.7,6.27)})
# sns.heatmap(mult)

In [ ]:
#
from collections import defaultdict
y2p = defaultdict(list)
for y,p in list(zip(*np.where(data.T >= 1))):
    y2p[y].append(p)


In [ ]:
data

In [ ]:
from itertools import combinations
import scipy.spatial as sp
results = []
for i in y2p.keys():
    combs = list(combinations(y2p[i], 2))
    # check similarity within the year
    for c in combs:    
        idx1 = metadata[(metadata.platform == data.index[c[0]]) & (metadata.year_int == data.columns[i]) & (metadata.modification_recoded == 1)].index
        idx2 = metadata[(metadata.platform == data.index[c[1]]) & (metadata.year_int == data.columns[i]) & (metadata.modification_recoded == 1)].index
        embeddings1= embeddings[list(idx1)]
        embeddings2= embeddings[list(idx2)]
        mult_comp = 1 - sp.distance.cdist(embeddings1, embeddings2, 'cosine')
        results.append([i, c[0], i,c[1], mult_comp.reshape(-1).max()])
    if i < len(y2p.keys()) - 1:
        for p in y2p[i]:
            idx1 = metadata[(metadata.platform == data.index[p]) & (metadata.year_int == data.columns[i]) & (metadata.modification_recoded == 1)].index
            for p2 in y2p[i+1]:
                idx2 = metadata[(metadata.platform == data.index[p2]) & (metadata.year_int == data.columns[i+1]) & (metadata.modification_recoded == 1)].index
                embeddings1= embeddings[list(idx1)]
                embeddings2= embeddings[list(idx2)]
                mult_comp = 1 - sp.distance.cdist(embeddings1, embeddings2, 'cosine')
                results.append([i, p, i+1, p2, mult_comp.reshape(-1).max()])
        
    

In [ ]:
results_filtered = [r for r in results if r[4] >= 0.9]

In [ ]:
results_filtered

In [ ]:
import plotly.graph_objects as go

In [ ]:
from itertools import product
nodes = list(product(range(data.shape[0]), range(data.shape[1])))
#nodes_label = list(range(len(nodes)))
nodes_label = [f'{data.index[n[0]]} - {data.columns[n[1]]}' for n in nodes]
x = [n[1]/data.shape[1] for n in nodes]
y = [n[0]/data.shape[0] for n in nodes]

In [ ]:
nodes = []
for r in results_filtered:
    nodes.append((r[1], r[0]))
    nodes.append((r[3], r[2]))
nodes_label = list(range(len(nodes)))
nodes_hover_label = [f'{data.index[n[0]]} - {data.columns[n[1]]}' for n in nodes]
x = [n[1]/data.shape[1] for n in nodes]
y = [n[0]/data.shape[0] for n in nodes]

In [ ]:
nodes2idx = {n: i for i, n in enumerate(nodes)}
nodes2idx

In [ ]:

source, target, value = [], [], []
for r in results_filtered:
    source.append(nodes2idx[(r[1], r[0])])
    target.append(nodes2idx[(r[3], r[2])])
    value.append(r[4])

In [ ]:
source, target

In [ ]:


fig = go.Figure(go.Sankey(
    arrangement = "snap",
    node = {
        "label": nodes_label,
        
        "x": x,
        "y": y,
        'customdata' : nodes_hover_label ,
      'hovertemplate':'%{customdata}',
        'pad':10},  # 10 Pixels
    link = {
        "source": source,
        "target": target,
        "value": value},
        )
        
    
)
fig.show(figsize=(20,20))

In [ ]:
import plotly.express as px

fig.write_html("figures/sankey_test.html")

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE


def plot_tsne(
    metadata: pd.DataFrame,
    embeddings: np.ndarray,
    color_by: str = "platform",       # "platform" | "year"
    perplexity: float = 30.0,
    n_iter: int = 1000,
    random_state: int = 42,
    title: str = "t-SNE projection of sentence embeddings",
) -> go.Figure:
    """
    Reduce embeddings to 2D with t-SNE and plot as an interactive scatter.

    Parameters
    ----------
    metadata     : DataFrame with 'year' and 'platform' columns.
    embeddings   : ndarray (N, D).
    color_by     : Column to use for point colour — 'platform' or 'year'.
    perplexity   : t-SNE perplexity. Rule of thumb: 5–50, scales with N.
    n_iter       : Number of optimisation iterations (≥250).
    random_state : Seed for reproducibility.
    title        : Figure title.
    """
    # 1. Fit t-SNE
    tsne = TSNE(
        n_components=3,
        perplexity=perplexity,
        n_iter=n_iter,
        random_state=random_state,
        init="pca",          # PCA init is more stable than random
        learning_rate="auto",
    )
    coords = tsne.fit_transform(embeddings)   # (N, 2)

    # 2. Build a plot-ready DataFrame
    plot_df = metadata[["year", "platform","sentence"]].copy().reset_index(drop=True)
    plot_df["x"] = coords[:, 0]
    plot_df["y"] = coords[:, 1]
    plot_df["z"] = coords[:, 2]  # Add z coordinate for 3D plotting
    plot_df["year"] = plot_df["year"].astype(str)   # treat year as category

    # 3. Scatter plot
    fig = px.scatter_3d(
        plot_df,
        x="x",
        y="y",
        z="z",  # Add a third dimension
        color=color_by,
        hover_data={"x": False, "y": False, "z": False, "platform": True, "year": True, "sentence": True},
        title=title,
        labels={"x": "t-SNE 1", "y": "t-SNE 2", "z": "t-SNE 3"},
        #color_discrete_sequence=PLATFORM_COLORS,
        opacity=0.75,
    )

    fig.update_traces(marker=dict(size=6, line=dict(width=0.3, color="white")))
    fig.update_layout(
        plot_bgcolor="white",
        paper_bgcolor="white",
        legend_title=color_by.capitalize(),
        margin=dict(l=40, r=40, t=60, b=40),
        width=750,
        height=550,
    )
    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False)

    return fig

In [ ]:
type(embeddings)

In [ ]:
metadata_selected = metadata[metadata.modification_recoded == 1]
metadata_selected = metadata_selected.dropna(subset=['platform', 'sentence'])
embeddings_selected = embeddings[metadata_selected.index]

In [ ]:
# Colour points by platform
fig = plot_tsne(metadata_selected, embeddings_selected, color_by="platform")
fig.show()



In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from tools.modification_sankey import build_sankey

In [ ]:
embeddings = np.loadtxt('processed_data/tous/embedding.tsv')

In [ ]:
metadata.shape, embeddings.shape

In [ ]:
build_sankey(metadata, embeddings)

## Fin